In [1]:
# Load the IPython autoreload extension
%load_ext autoreload
# Reload imported modules before each cell runs
%autoreload 2

ModuleNotFoundError: No module named 'autoreload  # Load the IPython autoreload extension'

In [ ]:
import math  # Math helpers (ceil, etc.) for subplot layout

import matplotlib.pyplot as plt  # Plotting library for scatter visualizations
import numpy as np  # Numerical arrays and helpers
import torch  # PyTorch tensors and model loading
from tqdm.auto import tqdm  # Progress bar for reverse sampling loops

import ddpm  # Local DDPM model, noise scheduler, and training entrypoints
import datasets  # Local 2D toy datasets (dino, moons, etc.)

### forward

In [ ]:
num_timesteps = 50  # Total diffusion steps in the forward process
plot_step = 5  # Plot a frame every this many timesteps

num_plots = math.ceil(num_timesteps / plot_step)  # How many subplot panels we need
num_cols = 5  # Number of subplot columns
num_rows = math.ceil(num_plots / num_cols)  # Number of subplot rows

fig = plt.figure(figsize=(15, 6))  # Create the forward-process figure

noise_scheduler = ddpm.NoiseScheduler(num_timesteps=num_timesteps)  # Build the noise schedule
dataset = datasets.get_dataset("dino", n=1000)  # Load 1000 dino 2D points
x0 = dataset.tensors[0]  # Clean data tensor of shape (N, 2)

plt_cnt = 1  # Subplot index for the clean data panel
plt.subplot(num_rows, num_cols, plt_cnt)  # Select the first subplot
plt.scatter(x0[:, 0], x0[:, 1], alpha=0.5, s=15)  # Plot clean data points
plt.title("data")  # Label the clean distribution
plt.xlim(-3.5, 3.5)  # Fix x-axis range for consistent panels
plt.ylim(-4., 4.75)  # Fix y-axis range for consistent panels
plt.axis("off")  # Hide axes for a cleaner look
        
for t in range(len(noise_scheduler)):  # Iterate over every diffusion timestep
    timesteps = np.repeat(t, len(x0))  # Same timestep index for all points
    noise = torch.randn_like(x0)  # Sample Gaussian noise matching x0 shape
    sample = noise_scheduler.add_noise(x0, noise, timesteps)  # Apply forward noising at step t
    if (t + 1) % plot_step == 0 and (t + 1) != len(noise_scheduler):  # Plot every plot_step (skip final)
        plt_cnt += 1  # Move to the next subplot index
        plt.subplot(num_rows, num_cols, plt_cnt)  # Select the next subplot
        plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15)  # Plot noised points
        plt.title(f"step: {t + 1}")  # Show which diffusion step this is
        plt.xlim(-3.5, 3.5)  # Keep x limits aligned across panels
        plt.ylim(-4., 4.75)  # Keep y limits aligned across panels
        plt.axis("off")  # Hide axes
        
fig.tight_layout()  # Reduce wasted whitespace between subplots
plt.savefig("static/forward.png", facecolor="white")  # Save the forward-process figure
plt.show()  # Display the figure in the notebook

### reverse

In [ ]:
!python ddpm.py --experiment_name dino_base

In [ ]:
model = ddpm.MLP()  # Instantiate the default MLP denoiser

path = "exps/dino_base/model.pth"  # Path to the trained baseline weights
model.load_state_dict(torch.load(path))  # Load checkpoint parameters into the model
model.eval()  # Switch to eval mode (disables dropout/etc. if present)

In [ ]:
eval_batch_size = 1000  # Number of points to generate during reverse sampling
num_timesteps = 50  # Diffusion steps used by the trained noise schedule
plot_step = 5  # Store a sample snapshot every this many reverse steps
noise_scheduler = ddpm.NoiseScheduler(num_timesteps=num_timesteps)  # Rebuild matching noise schedule
sample = torch.randn(eval_batch_size, 2)  # Start from pure Gaussian noise in 2D
timesteps = list(range(num_timesteps))[::-1]  # Reverse timesteps: T-1, ..., 0
samples = []  # Collected intermediate reverse samples for plotting
steps = []  # Matching step indices for subplot titles
for i, t in enumerate(tqdm(timesteps)):  # Denoise one reverse step at a time
    t = torch.from_numpy(np.repeat(t, eval_batch_size)).long()  # Broadcast scalar t to the batch
    with torch.no_grad():  # No gradients needed during sampling
        residual = model(sample, t)  # Predict noise residual at current timestep
    sample = noise_scheduler.step(residual, t[0], sample)  # Take one DDPM reverse step
    if (i + 1) % plot_step == 0:  # Keep every plot_step-th intermediate result
        samples.append(sample.numpy())  # Store numpy copy for matplotlib
        steps.append(i + 1)  # Record which reverse step this snapshot is from

In [ ]:
num_cols = 5  # Number of subplot columns for reverse panels
num_rows = math.ceil(len(samples) / num_cols)  # Rows needed to fit all snapshots
fig = plt.figure(figsize=(15, 6))  # Create the reverse-process figure
for i, sample in enumerate(samples):  # Plot each saved reverse snapshot
    plt.subplot(num_rows, num_cols, i + 1)  # Place snapshot i in the grid
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15)  # Draw generated 2D points
    plt.title(f"step: {steps[i]}")  # Label with reverse step index
    plt.xlim(-3.5, 3.5)  # Keep x limits consistent with forward plots
    plt.ylim(-4., 4.75)  # Keep y limits consistent with forward plots
    plt.axis("off")  # Hide axes
fig.tight_layout()  # Compact subplot spacing
plt.savefig("static/reverse.png", facecolor="white")  # Save reverse-process figure
plt.show()  # Show the figure in the notebook

## ablations

In [ ]:
def plot_ablation(frames_dict, outname):  # Plot ablation rows from saved training frames
    num_rows = len(frames_dict)  # One row per ablation setting/name
    num_cols = 10  # Number of epoch snapshot columns

    fig = plt.figure(figsize=(3.5*num_cols, 3*num_rows + 0.5))  # Size figure by grid shape
    row = 0  # Current row index among ablation settings

    for name, frames in frames_dict.items():  # Iterate over each ablation series
        epoch_step = len(frames) // num_cols  # Stride through saved epochs
        offset = row*(num_cols + 1)  # Flat subplot offset including label column
        plt.subplot(num_rows, num_cols + 1, offset + 1)  # Leftmost cell holds the row label
        plt.scatter(0, 0, alpha=0)  # Invisible point so text has a drawable axes
        plt.text(0, 0, name, fontdict={"size": 30})  # Write ablation setting name
        plt.xlim(-0.25, 2)  # Give the label some horizontal room
        plt.axis("off")  # Hide axes for the label cell

        for i in range(num_cols):  # Fill epoch snapshot columns
            plt.subplot(num_rows, num_cols + 1, offset + i + 2)  # Select snapshot subplot
            ix = i * epoch_step  # Epoch index to visualize in this column
            frame = frames[ix]  # Points generated/saved at that epoch
            plt.scatter(frame[:, 0], frame[:, 1], s=5, alpha=0.7)  # Plot the frame points
            if row == 0:  # Only the top row needs column titles
                if i == 0:  # First column includes the word "epoch"
                    title = f"epoch {ix}"
                else:  # Later columns just show the epoch number
                    title = f"{ix}"
                plt.title(title, fontdict={"size": 30}, pad=30)  # Large column title
            plt.xlim(-3.5, 3.5)  # Shared x limits across ablation panels
            plt.ylim(-4., 4.75)  # Shared y limits across ablation panels
            plt.axis("off")  # Hide axes

        row += 1  # Advance to the next ablation row

    plt.tight_layout()  # Compact spacing before saving
    plt.savefig(outname, facecolor="white")  # Write figure to the given path
    plt.show()  # Display the ablation grid

### datasets

In [ ]:
!python ddpm.py --dataset moons --experiment_name moons_base  # Train on moons dataset
!python ddpm.py --dataset dino --experiment_name dino_base  # Train on dino dataset
!python ddpm.py --dataset line --experiment_name line_base  # Train on line dataset
!python ddpm.py --dataset circle --experiment_name circle_base  # Train on circle dataset

In [ ]:
frames_dict = {  # Map dataset name -> saved training frames
    "moons": np.load("exps/moons_base/frames.npy"),  # Load moons experiment frames
    "dino": np.load("exps/dino_base/frames.npy"),  # Load dino experiment frames
    "line": np.load("exps/line_base/frames.npy"),  # Load line experiment frames
    "circle": np.load("exps/circle_base/frames.npy"),  # Load circle experiment frames
}

plot_ablation(frames_dict, "static/datasets.png")  # Render and save dataset ablation figure

### learning rate

In [ ]:
!python ddpm.py --learning_rate 1e-2 --experiment_name dino_lr1e-2  # Train with lr=1e-2
!python ddpm.py --learning_rate 1e-3 --experiment_name dino_lr1e-3  # Train with lr=1e-3
!python ddpm.py --learning_rate 1e-4 --experiment_name dino_lr1e-4  # Train with lr=1e-4
!python ddpm.py --learning_rate 1e-5 --experiment_name dino_lr1e-5  # Train with lr=1e-5

In [ ]:
frames_dict = {  # Map learning-rate label -> saved training frames
    "lr 1e-2": np.load("exps/dino_lr1e-2/frames.npy"),  # Frames from lr=1e-2 run
    "lr 1e-3": np.load("exps/dino_lr1e-3/frames.npy"),  # Frames from lr=1e-3 run
    "lr 1e-4": np.load("exps/dino_lr1e-4/frames.npy"),  # Frames from lr=1e-4 run
    "lr 1e-5": np.load("exps/dino_lr1e-5/frames.npy"),  # Frames from lr=1e-5 run
}

plot_ablation(frames_dict, "static/learning_rate.png")  # Render and save lr ablation figure

### num_timesteps

In [ ]:
!python ddpm.py --num_timesteps 5 --experiment_name dino_timesteps5  # Train with 5 diffusion steps
!python ddpm.py --num_timesteps 10 --experiment_name dino_timesteps10  # Train with 10 diffusion steps
!python ddpm.py --num_timesteps 25 --experiment_name dino_timesteps25  # Train with 25 diffusion steps
!python ddpm.py --num_timesteps 50 --experiment_name dino_timesteps50  # Train with 50 diffusion steps
!python ddpm.py --num_timesteps 100 --experiment_name dino_timesteps100  # Train with 100 diffusion steps
!python ddpm.py --num_timesteps 250 --experiment_name dino_timesteps250  # Train with 250 diffusion steps

In [ ]:
frames_dict = {  # Map padded timestep labels -> saved training frames
    "        5": np.load("exps/dino_timesteps5/frames.npy"),  # Frames from 5-step run
    "       10": np.load("exps/dino_timesteps10/frames.npy"),  # Frames from 10-step run
    "       25": np.load("exps/dino_timesteps25/frames.npy"),  # Frames from 25-step run
    "       50": np.load("exps/dino_timesteps50/frames.npy"),  # Frames from 50-step run
    "      100": np.load("exps/dino_timesteps100/frames.npy"),  # Frames from 100-step run
    "      250": np.load("exps/dino_timesteps250/frames.npy"),  # Frames from 250-step run
}

plot_ablation(frames_dict, "static/num_timesteps.png")  # Render and save timestep ablation figure

### beta schedule

In [ ]:
!python ddpm.py --beta_schedule quadratic --experiment_name dino_quadratic_schedule

In [ ]:
frames_dict = {  # Compare linear vs quadratic beta schedules
    "linear": np.load("exps/dino_base/frames.npy"),  # Baseline linear schedule frames
    "quadratic": np.load("exps/dino_quadratic_schedule/frames.npy"),  # Quadratic schedule frames
}

plot_ablation(frames_dict, "static/beta_schedule.png")  # Render and save schedule ablation figure

### hidden size

In [ ]:
!python ddpm.py --hidden_size 16 --experiment_name dino_hid_size_16  # Train MLP with hidden size 16
!python ddpm.py --hidden_size 32 --experiment_name dino_hid_size_32  # Train MLP with hidden size 32
!python ddpm.py --hidden_size 64 --experiment_name dino_hid_size_64  # Train MLP with hidden size 64
!python ddpm.py --hidden_size 256 --experiment_name dino_hid_size_256  # Train MLP with hidden size 256
!python ddpm.py --hidden_size 512 --experiment_name dino_hid_size_512  # Train MLP with hidden size 512

In [ ]:
frames_dict = {  # Map hidden-size labels -> saved training frames
    "       16": np.load("exps/dino_hid_size_16/frames.npy"),  # Frames from hidden_size=16
    "       32": np.load("exps/dino_hid_size_32/frames.npy"),  # Frames from hidden_size=32
    "       64": np.load("exps/dino_hid_size_64/frames.npy"),  # Frames from hidden_size=64
    "      128": np.load("exps/dino_base/frames.npy"),  # Baseline uses hidden_size=128
    "      256": np.load("exps/dino_hid_size_256/frames.npy"),  # Frames from hidden_size=256
    "      512": np.load("exps/dino_hid_size_512/frames.npy"),  # Frames from hidden_size=512
}

plot_ablation(frames_dict, "static/hidden_size.png")  # Render and save hidden-size ablation figure

### num layers

In [ ]:
!python ddpm.py --hidden_layers 1 --experiment_name dino_hid_layers_1  # Train MLP with 1 hidden layer
!python ddpm.py --hidden_layers 2 --experiment_name dino_hid_layers_2  # Train MLP with 2 hidden layers
!python ddpm.py --hidden_layers 4 --experiment_name dino_hid_layers_4  # Train MLP with 4 hidden layers
!python ddpm.py --hidden_layers 5 --experiment_name dino_hid_layers_5  # Train MLP with 5 hidden layers

In [ ]:
frames_dict = {  # Map layer-count labels -> saved training frames
    "        1": np.load("exps/dino_hid_layers_1/frames.npy"),  # Frames from 1-layer run
    "        2": np.load("exps/dino_hid_layers_2/frames.npy"),  # Frames from 2-layer run
    "        3": np.load("exps/dino_base/frames.npy"),  # Baseline uses 3 hidden layers
    "        4": np.load("exps/dino_hid_layers_4/frames.npy"),  # Frames from 4-layer run
    "        5": np.load("exps/dino_hid_layers_5/frames.npy"),  # Frames from 5-layer run
    
}

plot_ablation(frames_dict, "static/num_hidden_layers.png")  # Render and save depth ablation figure

### positional embedding (timestep)

In [ ]:
!python ddpm.py --time_embedding learnable --experiment_name dino_time_emb_learnable
!python ddpm.py --time_embedding linear --experiment_name dino_time_emb_linear
!python ddpm.py --time_embedding zero --experiment_name dino_time_emb_zeros

In [ ]:
frames_dict = {  # Compare timestep positional embedding variants
    "learnable": np.load("exps/dino_time_emb_learnable/frames.npy"),  # Learnable embedding frames
    "sinusoidal": np.load("exps/dino_base/frames.npy"),  # Baseline sinusoidal time embedding
    "linear": np.load("exps/dino_time_emb_linear/frames.npy"),  # Linear embedding frames
    "zero": np.load("exps/dino_time_emb_zeros/frames.npy"),  # Zero embedding frames
}

plot_ablation(frames_dict, "static/time_embedding.png")  # Render and save time-embedding ablation figure

### positional embedding (inputs)

In [ ]:
!python ddpm.py --input_embedding learnable --experiment_name dino_input_emb_learnable
!python ddpm.py --input_embedding linear --experiment_name dino_input_emb_linear
!python ddpm.py --input_embedding identity --experiment_name dino_input_emb_identity

In [ ]:
frames_dict = {  # Compare input positional embedding variants
    "learnable": np.load("exps/dino_input_emb_learnable/frames.npy"),  # Learnable input embedding frames
    "sinusoidal": np.load("exps/dino_base/frames.npy"),  # Baseline sinusoidal input embedding
    "linear": np.load("exps/dino_input_emb_linear/frames.npy"),  # Linear input embedding frames
    "identity": np.load("exps/dino_input_emb_identity/frames.npy"),  # Identity input embedding frames
}

plot_ablation(frames_dict, "static/input_embedding.png")  # Render and save input-embedding ablation figure

### forward and reverse process animation

In [ ]:
from celluloid import Camera  # Camera helper for building matplotlib animations

In [ ]:
num_timesteps = 250  # Longer schedule for a smoother animation
noise_scheduler = ddpm.NoiseScheduler(num_timesteps=num_timesteps)  # Build 250-step noise schedule

model = ddpm.MLP()  # Instantiate the denoiser MLP
path = "exps/dino_timesteps250/model.pth"  # Checkpoint trained with 250 timesteps
model.load_state_dict(torch.load(path))  # Load trained weights
model.eval()  # Disable training-time behavior

dataset = datasets.get_dataset("dino", n=1000)  # Load dino points for forward animation
x0 = dataset.tensors[0]  # Clean data tensor of shape (N, 2)

In [ ]:
forward_samples = []
forward_samples.append(x0)
for t in range(len(noise_scheduler)):
    timesteps = np.repeat(t, len(x0))
    noise = torch.randn_like(x0)
    sample = noise_scheduler.add_noise(x0, noise, timesteps)
    forward_samples.append(sample)

In [ ]:
eval_batch_size = len(dataset)  # Generate one reverse particle per dataset point
sample = torch.randn(eval_batch_size, 2)  # Start reverse process from pure noise
timesteps = list(range(num_timesteps))[::-1]  # Reverse order: T-1 down to 0
reverse_samples = []  # Store one frame per reverse step
reverse_samples.append(sample.numpy())  # Frame 0 is the initial noise
for i, t in enumerate(tqdm(timesteps)):  # Denoise step by step with a progress bar
    t = torch.from_numpy(np.repeat(t, eval_batch_size)).long()  # Broadcast timestep to batch
    with torch.no_grad():  # Inference only; skip autograd
        residual = model(sample, t)  # Predict noise residual at timestep t
    sample = noise_scheduler.step(residual, t[0], sample)  # Apply one reverse diffusion step
    reverse_samples.append(sample.numpy())  # Keep frame for the animation

In [ ]:
xmin, xmax = -3.5, 3.5  # Shared x-axis limits for animation frames
ymin, ymax = -4., 4.75  # Shared y-axis limits for animation frames

fig, ax = plt.subplots()  # Create a single axes used by every animation frame
camera = Camera(fig)  # Wrap the figure so each snap() becomes a video frame

# forward
for i, sample in enumerate(forward_samples):  # Animate the forward noising process
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15, color="blue")  # Draw current points
    ax.text(0.0, 0.95, f"step {i: 4} / {num_timesteps}", transform=ax.transAxes)  # Step counter label
    ax.text(0.0, 1.01, "Forward process", transform=ax.transAxes, size=15)  # Process title
    plt.xlim(xmin, xmax)  # Lock x limits so the camera does not jump
    plt.ylim(ymin, ymax)  # Lock y limits so the camera does not jump
    plt.axis("off")  # Hide axes for a cleaner video
    camera.snap()  # Capture this frame
        
# reverse
for i, sample in enumerate(reverse_samples):  # Animate the reverse denoising process
    plt.scatter(sample[:, 0], sample[:, 1], alpha=0.5, s=15, color="blue")  # Draw current points
    ax.text(0.0, 0.95, f"step {i: 4} / {num_timesteps}", transform=ax.transAxes)  # Step counter label
    ax.text(0.0, 1.01, "Reverse process", transform=ax.transAxes, size=15)  # Process title
    plt.xlim(xmin, xmax)  # Keep x limits consistent with forward frames
    plt.ylim(ymin, ymax)  # Keep y limits consistent with forward frames
    plt.axis("off")  # Hide axes
    camera.snap()  # Capture this frame
    
animation = camera.animate(blit=True, interval=35)  # Build the animation (~35ms per frame)
animation.save("static/animation.mp4")  # Export the combined forward/reverse video